## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Clinical Text Analysis model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *clinical_text_analysis* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).

## Pipeline to Extract Clinical Entities, Assign Assertion Status and Find Relations

Overview:

Clinical entity detection, assertion status assignment, and relation extraction are crucial for analyzing medical texts. These processes can be incredibly useful to healthcare professionals, researchers, and medical NLP practitioners, enabling the extraction of valuable insights from clinical literature, electronic health records, and patient notes.

- **Model**: [en.explain_doc.clinical_granular](https://nlp.johnsnowlabs.com/2024/05/07/explain_clinical_doc_granular_en.html)
- **Model Description**: A pipeline designed for extracting clinical entities using the `ner_jsl` NER model, assigning their assertion status with the `assertion_jsl` model, and establishing relationships between the extracted entities with the `re_test_result_date` relation extraction model.

In [ ]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**: 
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }

    ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
    {
        "text": "Single text document"
    }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.

### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record, ensure_ascii=False) if content_type == "application/json" else record,
      
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return [json.loads(line) for line in response_body.strip().split('\n') if line]
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

Found the config file in: ./config.json


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.

In [ ]:
docs = [
    """The patient is a 21-day-old Caucasian male here for 2 days of congestion - mom has been suctioning yellow discharge from the patient's nares, plus she has noticed some mild problems with his breathing while feeding (but negative for any perioral cyanosis or retractions). Additionally, there is no side effect observed after Influenza vaccine. One day ago, mom also noticed a tactile temperature and gave the patient Tylenol. Baby also has had some decreased p.o. intake. His normal breast-feeding is down from 20 minutes q.2h. to 5 to 10 minutes secondary to his respiratory congestion. He sleeps well, but has been more tired and has been fussy over the past 2 days. The parents noticed no improvement with albuterol treatments given in the ER. His urine output has also decreased; normally he has 8 to 10 wet and 5 dirty diapers per 24 hours, now he has down to 4 wet diapers per 24 hours. Mom denies any diarrhea. His bowel movements are yellow colored and soft in nature.""",
    """The patient is a 40-year-old white male who presents with a chief complaint of "chest pain". The patient is diabetic and has a prior history of coronary artery disease. The patient presents today stating that his chest pain started yesterday evening and has been somewhat intermittent. He has been advised Aspirin 81 milligrams QDay, insulin 50 units in a.m. HCTZ 50 mg QDay. Nitroglycerin 1/150 sublingually PRN chest pain."""
]


sample_text = "The patient admitted for gastrointestinal pathology, under working treatment. History of prior heart murmur with echocardiogram findings as above on March 1998. According to the latest echocardiogram, basically revealed normal left ventricular function with left atrial enlargement . Based on the above findings, we will treat her medically with ACE inhibitors and diuretics and see how she fares."

### JSON


In [8]:
input_json_data = {"text": sample_text}
data = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
data["predictions"]

{'ner_predictions': [{'ner_chunk': 'admitted',
   'begin': 12,
   'end': 19,
   'ner_label': 'Admission_Discharge',
   'ner_confidence': '0.9993'},
  {'ner_chunk': 'gastrointestinal pathology',
   'begin': 25,
   'end': 50,
   'ner_label': 'Clinical_Dept',
   'ner_confidence': '0.7198'},
  {'ner_chunk': 'heart murmur',
   'begin': 95,
   'end': 106,
   'ner_label': 'Heart_Disease',
   'ner_confidence': '0.5289'},
  {'ner_chunk': 'echocardiogram',
   'begin': 113,
   'end': 126,
   'ner_label': 'Test',
   'ner_confidence': '0.9935'},
  {'ner_chunk': 'March 1998',
   'begin': 149,
   'end': 158,
   'ner_label': 'Date',
   'ner_confidence': '0.874'},
  {'ner_chunk': 'echocardiogram',
   'begin': 185,
   'end': 198,
   'ner_label': 'Test',
   'ner_confidence': '0.9402'},
  {'ner_chunk': 'normal',
   'begin': 220,
   'end': 225,
   'ner_label': 'Test_Result',
   'ner_confidence': '0.4029'},
  {'ner_chunk': 'left ventricular function',
   'begin': 227,
   'end': 251,
   'ner_label': 'Test',


### JSON Lines


In [9]:
def create_jsonl(records):
    if isinstance(records, str):
        records = [records]
    json_records = [{"text": text} for text in records]
    json_lines = "\n".join(json.dumps(record) for record in json_records)
    return json_lines


In [14]:
input_jsonl_data = create_jsonl(docs)
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
[print(prediction) for prediction in data]

{'predictions': {'ner_predictions': [{'ner_chunk': '21-day-old', 'begin': 17, 'end': 26, 'ner_label': 'Age', 'ner_confidence': '0.997'}, {'ner_chunk': 'Caucasian', 'begin': 28, 'end': 36, 'ner_label': 'Race_Ethnicity', 'ner_confidence': '0.9995'}, {'ner_chunk': 'male', 'begin': 38, 'end': 41, 'ner_label': 'Gender', 'ner_confidence': '0.9998'}, {'ner_chunk': '2 days', 'begin': 52, 'end': 57, 'ner_label': 'Duration', 'ner_confidence': '0.805'}, {'ner_chunk': 'congestion', 'begin': 62, 'end': 71, 'ner_label': 'Symptom', 'ner_confidence': '0.9049'}, {'ner_chunk': 'mom', 'begin': 75, 'end': 77, 'ner_label': 'Gender', 'ner_confidence': '0.9907'}, {'ner_chunk': 'suctioning yellow discharge', 'begin': 88, 'end': 114, 'ner_label': 'Symptom', 'ner_confidence': '0.26813334'}, {'ner_chunk': 'nares', 'begin': 135, 'end': 139, 'ner_label': 'External_body_part_or_region', 'ner_confidence': '0.7284'}, {'ner_chunk': 'she', 'begin': 147, 'end': 149, 'ner_label': 'Gender', 'ner_confidence': '0.9978'}, {'

[None, None]